In [ ]:
from dataclasses import dataclass
from typing import List
import math
import random

@dataclass
class Bucket:
    size: int      # number of 1s this bucket represents
    end_time: int  # timestamp of the most recent 1 in this bucket

class DGIM:
    def __init__(self, window_size: int):
        if window_size <= 0:
            raise ValueError("window_size must be positive")
        self.W = window_size
        self.time = 0
        self.buckets: List[Bucket] = []  # newest first

    def _expire(self):
        """Drop buckets that are completely outside the sliding window."""
        boundary = self.time - self.W
        self.buckets = [b for b in self.buckets if b.end_time > boundary]

    def _compress(self):
        """Ensure ≤2 buckets per size by merging the oldest ones if necessary."""
        i = 0
        while i < len(self.buckets):
            size_i = self.buckets[i].size
            j = i
            count = 0
            while j < len(self.buckets) and self.buckets[j].size == size_i:
                count += 1
                j += 1
            while count > 2:
                b_oldest = self.buckets.pop(j-1)
                b_second_oldest = self.buckets.pop(j-2)
                merged = Bucket(size=b_oldest.size * 2, end_time=b_second_oldest.end_time)
                self.buckets.insert(j-2, merged)
                # restart check from here
                i = max(0, i-1)
                break
            else:
                i = j

    def add(self, bit: int):
        """Add the next bit from the stream."""
        if bit not in (0, 1):
            raise ValueError("bit must be 0 or 1")
        self.time += 1
        if bit == 1:
            self.buckets.insert(0, Bucket(size=1, end_time=self.time))
            self._compress()
        self._expire()

    def query(self, k: int) -> int:
        """Estimate the number of 1s in the last k bits (k ≤ W)."""
        if k <= 0:
            return 0
        if k > self.W:
            k = self.W
        threshold = self.time - k
        total = 0
        partial_added = False
        for b in self.buckets:
            if b.end_time <= threshold:
                continue
            if not partial_added:
                oldest_partial = b
                partial_added = True
            else:
                total += b.size
        if partial_added:
            total += math.ceil(oldest_partial.size / 2)
        return total


# ---------------- Demo ----------------
if __name__ == "__main__":
    dg = DGIM(window_size=100)
    stream = [random.choice([0,1]) for _ in range(200)]

    for bit in stream:
        dg.add(bit)

    k = 50
    approx = dg.query(k)
    exact = sum(stream[-k:])
    print(f"DGIM estimate of 1s in last {k} bits: {approx}")
    print(f"Exact count: {exact}")